# FormulaLM — Evaluate Fine-Tuned QLoRA v1 Adapter & Error Analysis

Loads the trained **QLoRA v1** adapter (`checkpoint-200`, best `eval_loss = 0.004240`) onto `Qwen/Qwen3-1.7B` in non-thinking mode (`temperature=0`, `max_tokens=96`, `seed=42`) and evaluates it on BOTH held-out test sets:
1. `results/ft_v1_real` — `data/test_real/test_real_DRAFT.csv` (80 hand-written realistic requests)
2. `results/ft_v1_synthetic` — `data/processed/test_synthetic.jsonl` (500 unseen-schema, unseen-phrasing examples)

Then compares **Base 0-shot vs Base 5-shot vs Fine-tuned v1** (overall, by category, by difficulty) and clusters all v1 failures into 6 actionable clusters with 5 examples each.

**Hardware:** Colab T4 GPU (~2–3 minutes total).

In [1]:
# 1. Verify CUDA T4 GPU and mount Google Drive (where qlora_v1/adapter was saved)
import pathlib, torch

assert torch.cuda.is_available(), 'No CUDA GPU detected! Switch kernel to T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped (will fall back to Hugging Face adapter if not on Drive):', exc)

In [2]:
# 2. Clone / pull FormulaLM, install eval deps, and ensure test_synthetic.jsonl exists
import getpass, os, pathlib, subprocess, sys

REPO_DIR = pathlib.Path('/content/FormulaLM')
if not REPO_DIR.exists():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            token = None
    if not token:
        token = getpass.getpass('GitHub PAT for abubakarp789/FormulaLM (hidden, empty if public): ').strip()
    url = f'https://{token}@github.com/abubakarp789/FormulaLM.git' if token else 'https://github.com/abubakarp789/FormulaLM.git'
    subprocess.run(['git', 'clone', url, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

%cd /content/FormulaLM
if '/content/FormulaLM/src' not in sys.path:
    sys.path.insert(0, '/content/FormulaLM/src')

%pip install -q formulas openpyxl pandas pyyaml requests tqdm accelerate peft "transformers>=4.51.0" "huggingface_hub>=0.30.0"

if not pathlib.Path('data/processed/test_synthetic.jsonl').is_file():
    !python -m src.data.build_dataset --workers 2

In [3]:
# 3. Load Qwen/Qwen3-1.7B + QLoRA v1 adapter and evaluate on BOTH test sets
import pathlib
from src.eval.run_eval import TransformersBackend, evaluate

MODEL_ID = 'Qwen/Qwen3-1.7B'
REVISION = '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'

# Auto-locate adapter: Drive -> local outputs -> Hugging Face Hub
CANDIDATE_ADAPTERS = [
    '/content/drive/MyDrive/FormulaLM/qlora_v1/adapter',
    'outputs/qlora_v1/adapter',
]
ADAPTER_PATH = next((p for p in CANDIDATE_ADAPTERS if pathlib.Path(p).is_dir()), 'abubakar56/formulalm-qwen3-1.7b-lora')
print('Loading adapter from:', ADAPTER_PATH, flush=True)

backend = TransformersBackend(
    MODEL_ID,
    adapter_path=ADAPTER_PATH,
    revision=REVISION,
    load_in_4bit=False,
    enable_thinking=False,
)

for run_name, dataset in [
    ('ft_v1_real', 'data/test_real/test_real_DRAFT.csv'),
    ('ft_v1_synthetic', 'data/processed/test_synthetic.jsonl'),
]:
    print(f'=== Running {run_name} ===', flush=True)
    m = evaluate(
        backend=backend,
        dataset_path=dataset,
        run_name=run_name,
        seed=42,
        max_tokens=96,
        few_shot=0,
        batch_size=8,
        resume=True,
    )
    ov = m['overall']
    print(
        f"{run_name}: n={ov['n']} | syntax={100*ov['syntax_valid_rate']:.1f}% | "
        f"exec_acc={100*ov['execution_accuracy']:.1f}% | "
        f"exact={100*ov['exact_match_rate']:.1f}% | "
        f"raw_fmt={100*ov['raw_format_ok_rate']:.1f}% | "
        f"avg_lat={ov['avg_latency_ms']:.1f}ms",
        flush=True,
    )

In [4]:
# 4. Produce full Base vs Fine-Tuned v1 comparison table + 6-cluster error analysis
!python -m src.eval.compare_and_cluster \
  --ft-synthetic results/ft_v1_synthetic \
  --ft-real results/ft_v1_real \
  --ft-label "Fine-tuned v1" \
  --examples-per-cluster 5 \
  --out results/ft_v1_comparison.md \
  --json-out results/ft_v1_comparison.json

In [5]:
# 5. Push lightweight evaluation results (results/ft_v1_*) straight back to GitHub
PUSH_TO_GITHUB = True
if PUSH_TO_GITHUB:
    !git config user.name "abubakarp789"
    !git config user.email "abubakarp789@gmail.com"
    !git add results/ft_v1_*
    !git commit -m "results: add QLoRA v1 evaluation and 6-cluster error analysis"
    !git pull --rebase origin main
    !git push origin main